In [ ]:
import logging
import time
from corriscope.fpga_array import FPGAArray
import numpy as np
# Must use %matplotlib widget or %matplotlib ipympl with jupyter-lab. Needs 'pip install ipympl'.  jupyter-notebook needs %matplotlib notebook
%matplotlib ipympl 
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (9.5, 6)

log=logging.getLogger()
log.setLevel(logging.DEBUG)

In [ ]:
# hwm = 'crs 37:1 39:2 48:3 55:4 28:5 30:6 26:7 23:8'
# hwm = 'crs 55:1 37:2 48:3 39:4 74:5 83:6 94:7 140:8 '
# hwm = 'crs 55:1'
# hwm=  'CRS 167:1 146:2 142:3 171:4 139:5 159:6 211:7 245:8 216:9 152:10 165:11 242:12 214:13 244:14 221:15 228:16'
hwm = 'crc4 1'
# hwm='crs 192.168.88.29'
# ca = FPGAArray(hwm, mode='corr32', sync_method='irigb', sync_source='bp_trig', stderr_log_level='warn')
ca = await FPGAArray(hwm, mode='corr64', sync_method='irigb', sync_source='bp_trig', sync_master=1, sync_master_source='sync_out', sync_master_output=1, stderr_log_level='info', ignore_missing_boards=True)
print(f'Using firmware with timestamp: {await ca.ib[0].get_fpga_firmware_timestamp_async()}')

In [ ]:
# i= ca.ib[slot]
raw_period=0.050
integ_period=32768 

# Data rates for 8 boards. 1Gb switch with 10 Gbps link to PC.
# raw= 25 ms, integ=16384: 2.35 Gbps. No UCAP/UCORR overruns. No loss.
# raw = 25 ms, integ=8192, 3.35 Gbps. No UCAP/UCORR overruns. Starting to lose packets/timestamps
# raw = 25 ms, integ=4096, 5.39 Gbps. No UCAP/UCORR overruns. Only first couple of timestamps can be processed. After that processor can't keep up with data rate 
# raw = 25 ms, integ=2048, 5.39 Gbps. UCORR overruns. UCAP ok. 

# Lets capture some raw FFT data so we can compare with the correlator output
# Start data capture for raw FFT            
await ca.start_data_capture_async(raw_period, source='scaler', mode=3, channels=[0], data_type='fft_2x', sync=False)
await ca.start_correlators_async(integ_period)

In [ ]:
# Optional, for debug: Sync the boards to make sure it works, although that was done automatically on init. Check the sync counters to make sure all boards sync'ed. 
await ca.sync_async()
ca.ib.REFCLK.SYNC_CTR

In [ ]:
# Get a raw data receiver
r = await ca.get_data_receiver_async(tcp=False)
cr = await ca.get_corr_receiver_async()


In [ ]:
# Set capture parameters
Ncap=10
gain=1<<25
# Setup channelizers
await ca.set_channelizers_async(data_source='adc', funcgen_left_shift=0, fft_bypass=False, offset_binary_encoding=False, scaler_bypass=False, gains=gain, sync=False)

In [ ]:
# Get double resolution raw FFT data. Manually remap the data (not yet done automatically)  and scale to the 4-bit data seen by the correlator.
slot = 0  # Slot from which we'l get the data. Will use channel 0 on that slot.
t,fft2,c = r.read_raw_frames(ncap=Ncap, split=True, verbose=0)
# frev = ca.ib[0].chan[0].FFT.get_bin_map(reverse=True) # get the reverse bin map
fft2 = fft2[:,:, :] * gain / 2**(48-4) # Reorder bins and scale to 4-bits to reproduce what the correlator sees
fft2_4bit = fft2.round()
fft2_4bin = fft2_4bit.real.clip(-7,7) + 1j * fft2_4bit.imag.clip(-7,7)

In [ ]:
# Get useful maps
ijmap=cr.get_ij_to_prod_map() # used to convert (i,j) into a linear product index 
in_map=cr.get_good_bin_mask()
# # COmpute a map indicating  which bin is expected to have valid data (Essentially flags the first 256 bins and the correlator sacrificial bins)
# in_map = np.zeros(8192, bool)
# in_map[cr.bin_map.flatten()]=True
# bad_bins = cr.bin_map[:,0,0] # indices of bins that are sacrificed by the correlator for their read cycle
# in_map[bad_bins]=False # Remove corr sacrificial bins

In [ ]:
# Show the aligner error flags - should be all zeros 

for b in ca.ib: # First Clear the aligner flags
    ct2 = b.CT.CT2
    ct3 = b.CT.CT3
    ct2.ALIGN_MON_RESET=1
    ct2.ALIGN_MON_RESET=0
    ct2.ALIGN_MON_SOURCE=0
    ct3.ALIGN_MON_RESET=1
    ct3.ALIGN_MON_RESET=0
    ct3.ALIGN_MON_SOURCE=0
time.sleep(0.3) # Wait for errors to accumulate
ct2_status = [b.CT.CT2.ALIGN_MON_WORD for b in ca.ib] 
ct3_status = [b.CT.CT3.ALIGN_MON_WORD for b in ca.ib] 
if any(ct2_status) or any(ct3_status):
    print(f'CT data shuffling errors: CT2: {ct2_status}, CT3: {ct3_status}')
else:
    print('CT Engine shuffling is OK')

In [ ]:
for b in ca.ib: # First Clear UCORR and UCAP overrun flags
    b.CORR.OVERRUN_RESET=1
    b.CORR.OVERRUN_RESET=0
    b.UCAP.USER_RESET=1
    b.UCAP.USER_RESET=0


In [ ]:
# Get correlator data. We integrate 8 frames so we get all bin sets, wich are automatically rotating by default on each integ period.
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush=True, soft_integ_period=8, number_of_results=2, verbose=1)
print(f'{tt0=}')
print(f'{cc0=}')

In [ ]:
ucap_overrun = [b.UCAP.OVERRUN for b in ca.ib]
ucorr_overrun = [b.CORR.OVERRUN for b in ca.ib]
if any(ucap_overrun) or any(ucorr_overrun):
    print(f'Overruns: UCAP : {ucap_overrun}, UCORR: {ucorr_overrun}')
else:
    print('No UCAP or UCORR overruns')
    


In [ ]:
# Plot the raw-FFT-based autocorrelation and the correlator output 
b=np.arange(8192)
fig, ax = plt.subplots(figsize=(16,8))
ax.set_title('Autocorrelations')
# mags_corr = abs(dd0[0,:,0])/1031*923

# ax.plot(bad_bins, mags_corr[bad_bins], 'r.', ms=10)
# ax.plot(mags,'k.', ms=3)
# ax.plot(b, adc_fft,'c.', ms=3, label='numpy fft')
# for ii in range(1,2):
#     for jj in range(1):
# chans = range(32) # (8,) a# (0,1,8,9,16,17,24,25)

chans = (0,) # list of channels whose autocorr will be plotted
corr_sample = 0
if cc0[corr_sample] != 126976:
    raise RuntimeError(f'We got only {cc0[corr_sample]} samples') 
for ii in chans:
        fft_mags= ((np.abs(fft2[ii]))**2).mean(axis=0)*integ_period # SImulte correlator operations
        fft_mags_4bit = ((np.abs(fft2_4bit[ii]))**2).mean(axis=0)*integ_period
        jj = ii
        prod = ijmap[ii,jj]
        corr_mags = abs(dd0[corr_sample,:,prod])/1031*923        # self.soft_integ_period = soft_integ_period,
        # self.skip_partial_integs = skip_partial_integs,

        # corr_mags = abs(dd0[:,:,prod]).sum(axis=0)/1031*923
        # if max(corr_mags[2100:4000]) < 10000:
        #     continue
        ax.plot(b[1:], fft_mags[1:],'k.', ms=8, label='16+16 FFT')
        ax.plot(b[1:], fft_mags_4bit[1:],'b.', ms=8, label='16+16 FFT -> 4bit')
        ax.plot(b[in_map], corr_mags[in_map] ,'g.', ms=7, label=f'corr ({ii},{jj}) (all data)')
        # for corr in range(8):
        #     im = np.zeros(8192, bool)
        #     im[cr.bin_map[:,corr,:,0]]=True
        #     im[cr.bin_map[:,:,0,0]]=False
        #     ax.plot(b[im], corr_mags[im] ,'.', ms=20, label=f'corr ({ii},{jj}) for corr {corr}')
        #     # ax.plot(b, corr_mags ,'.', ms=5, label=f'corr ({ii},{jj}) for corr {corr}')
# ax.set_ylim([0,2**17])
ax.grid()
ax.legend()

# End of notebook

In [ ]:
n=0
while True:
    s=tuple(int(b.CORR.OVERRUN) for b in ca.ib)
    if n % 1000==0: print('.', end='')
    n+=1
    # print(f'[{s}]' if any(s) else '.', end='') 
    # time.sleep(1)

In [ ]:
for b in ca.ib:
    print(b, "CORR.OVERRUN", int(b.CORR.OVERRUN),
          "UCAP.OVERRUN", int(b.UCAP.OVERRUN),
          "UCAP.FIFO_OVERFLOW", int(b.UCAP.FIFO_OVERFLOW),
          "UCAP.USER_RESET", int(b.UCAP.USER_RESET),
          "ucap_out", int(b.UCAP.OUT_FRAME_CTR),
          "corr_out", int(b.CORR.OUT_FRAME_CTR))

In [ ]:
for b in ca.ib:
    b.UCAP.USER_RESET=1

In [ ]:
ijmap=ca.ib[0].CORR.get_ij_to_prod_map()

In [ ]:
ijmap[1,1]

In [ ]:
for bix, b in enumerate(ca.ib):
    for cix,ch in enumerate(b.chan):
        if (bix,cix) != (0,0):
            ch.FUNCGEN.RIGHT_SHIFT=15

In [ ]:
ca.ib[0].chan[0].FUNCGEN.RIGHT_SHIFT=0

In [ ]:
bm[:,::4,0][:,47:56]

In [ ]:
sorted(*np.where(in_map))[:10]

In [ ]:
np.where(bm[:,::4,0]==259+4*5)

In [ ]:
fb = ca.ib[0].chan[0].FFT.

In [ ]:
fb[:,:12]

In [ ]:
m=np.meshgrid(range(10), range(3))

In [ ]:
np.arange(3)[None,:] + np.arange(10)[:, None]

In [ ]:
bins = np.arange(8192).reshape((4, -1), order='C')
bins = (bins - np.arange(2048) * 2048) % 8192
bins= bins.flatten(order='F')

In [ ]:
rb = np.empty(8192, dtype=np.intp)

In [ ]:
rb[bins] = np.arange(8192)

In [ ]:
rb

In [ ]:
frev2

In [ ]:
all(frev2 == rb)

In [ ]:
ca.ib[0].RX_TO_TX_LANE_MAP

In [ ]:
tx_to_rx = ca.ib[0].TX_TO_RX_LANE_MAP

In [ ]:
slot_to_lane = {(ss,ds):(sl,dl) for (ss,sl),(ds,dl) in tx_to_rx.items()}

In [ ]:
[[slot_to_lane[ss, ds][1] for ss in range(4)] for ds in range(4)] 

In [ ]:
await i.get_fpga_firmware_timestamp_async()

In [ ]:
for b in ca.ib:
    print(f'{b}: CT_LEVEL={b.CT.CT_LEVEL}, FFT_ROTATE={b.chan[0].FFT.is_rotated}, STREAM_ID={b.UCAP.USER_STREAM_ID*16}+channel')

In [ ]:
for b in ca.ib:
    print(f'{b}: STREAM_ID={b.UCAP.USER_STREAM_ID*16}+channel')

In [ ]:
ct=i.CT.

In [ ]:
bp = [i.CT.BPLINKS for  i in ca.ib]
y=[y for  i in ca.ib for y in i.CT.BPLINKS.gty] # list of all GTY transceivers in the array
i = ca.ib[0]
def get_monitor(bp, lane, source, data_type, n=1, reset=False): 
    """ Get BP Link monitoring data """
    bp.LANE_SEL=lane
    tx_sources = {'tx': 0, 'rxfifo': 1}  # sources that are on the TX clock, otherwise they are on the RX clock
    if source in tx_sources:
        bp.LANE_MON_SRC= tx_sources[source] 
        data_types = {'clk_ctr': 0, 'word_ctr':1, 'frame_ctr':2, 'frame_len': 3, 'frame_len_min':4, 'frame_len_max':5, 'err_ctr':6, 'fifo_ovf_ctr':7, 'delay':8, 'flags':9}
    elif source == 'rx':
        data_types = {'clk_ctr': 0, 'word_ctr':1, 'frame_ctr':2, 'frame_len': 3, 'frame_len_min':4, 'frame_len_max':5, 'err_ctr':6, 'delay':8, 'flags':9}
    else:
        raise RuntimeError('Invalid source')
    if data_type not in data_types:
        raise RuntimeError('Invalid data type')
    bp.LANE_MON_TYPE = data_types[data_type]
        
    if reset:
        # reset stats
        bp.RESET_STATS=1
        bp.RESET_STATS=0

    # Get the monitoring data
    if source in tx_sources:
        if n == 1:
            bp.LANE_MON_EN=1
            return bp.TXCLK_MON_WORD
        else:
            r = []
            for _ in range(n):
                bp.LANE_MON_EN=1
                r.append(bp.TXCLK_MON_WORD) 
                bp.LANE_MON_EN=0
            return r
    else:
        if n == 1:
            return bp.RXCLK_MON_WORD
        else:
            r = []
            for _ in range(n):
                bp.LANE_MON_EN=1
                r.append(bp.RXCLK_MON_WORD)
                bp.LANE_MON_EN=0
            return r


In [ ]:
N=500  # number of measurements 
expected = {
    'clk_ctr': 'Changes randomly because it\'s counting very fast', 
    'word_ctr': 'Counts from 0 to packet length -1', 
    'frame_ctr': 'Counts from 0 (after stats reset) to some number of frame, will wrap if wait long enough', 
    'frame_len': 'Current frame length minus one, updated at every frame. All measurements should be the same', 
    'frame_len_min': 'Smallest frame length since last stats reset (detects spurious bad frames). Should be the expected frame length', 
    'frame_len_max': 'Largest frame length since last stats reset (detects spurious bad frames). Should be the expected frame length', 
    'err_ctr':'Counts RX errors since last stats reset. Is zero on TX. Must be zero.', 
    'fifo_ovf_ctr':'Counts TX or RX FIFO overflows since last reset. Must be zeros',
    'flags': 'Current values of flags'
}

In [ ]:
y=[y for  b in ca.ib for y in b.CT.BPLINKS.gty] # list of all GTY transceivers in the array
for yy in y:
    yy.TXDIFFCTRL = 10
    yy.TXPRECURSOR = 5
    yy.TXPOSTCURSOR = 0

for yy in y:
    yy.reset_rx_equalizer()
    

In [ ]:
print('XXVGLINK_ARRAY monitoring information')
lane_names = ('CT2-0*', 'CT2-1', 'CT2-2', 'CT2-3', 'CT3-0*', 'CT3-1*', 'CT3-2', 'CT3-3')
source = 'rxfifo'
data_types = ('fifo_ovf_ctr','err_ctr', 'frame_ctr', 'frame_len')
NN=100
s2 = ' | '.join(f'{s:11s}' for s in lane_names)
print(f'{"":55s} | {s2} (*=direct)')
for b in bp:
    for data_type in data_types: # ,
        values = []
        for lane in range(8):
            d=get_monitor(b, lane, source, data_type, NN, reset=True)
            values.append((min(d), max(d), len(set(d))))
        s1 = (f'{vmin}-{vmax}' if vlen > 1 else f'{vmin}' for vmin, vmax, vlen in values)
        s2 = ' | '.join(f'{s:11s}' for s in s1)
        bname = f'{str(b) if data_type==data_types[0] else "":32s}'
        print(f'{bname}: {source} {data_type:15s}: {s2}')

In [ ]:
# print(f'Checking TX data sent by slot {tx_slot} to {rx_slot}')
for b in bp:
    for lane in range(0,4):
        # for data_type in ('clk_ctr', 'word_ctr', 'frame_ctr', 'frame_len', 'frame_len_min', 'frame_len_max',  'delay', 'fifo_ovf_ctr'): # ,
        for data_type in ('delay', 'fifo_ovf_ctr'): # ,
            d=get_monitor(b, lane, 'rxfifo', data_type, N, reset=True)
            print(f' {b} {lane}:   TX {data_type}: {len(set(d))} values between {min(d)} and {max(d)}')

In [ ]:
# print(f'Checking Post-FIFO RX data received by slot {rx_slot} from {tx_slot}')
for data_type in ('clk_ctr', 'word_ctr', 'frame_ctr', 'frame_len', 'frame_len_min', 'frame_len_max', 'err_ctr', 'fifo_ovf_ctr', 'flags'):
    d=get_monitor(bp[0], 0, 'tx', data_type, N)
    print(f'   RXFIFO {data_type}: {len(set(d))} values between {min(d)} and {max(d)}')

In [ ]:
a=bp[0]
a.LANE_MON_EN=1

In [ ]:
f'{a.TXCLK_MON_WORD:016b}'

In [ ]:
await ca.sync_async()

In [ ]:
for b in ca.ib:
    ct2 = b.CT.CT2
    ct2.set_lane_map([0,0,0,0],[0,0,0,0])

In [ ]:
for b in bp:
    for lane in range(0,4):
        d=get_monitor(b,lane, 'tx', 'delay', N, reset=0)
        print(f' {b} {lane}:  RXFIFO delay: {len(set(d))} values between {min(d)} and {max(d)}')

In [ ]:
i.FreqCtr.status()

In [ ]:
await ca.sync_async()

In [ ]:
for b in ca.ib:
    ct2 = b.CT.CT2
    for lane in range(4):
        ct2.ALIGN_MON_SOURCE=0
        ct2.ALIGN_MON_LANE=lane
        ct2.ALIGN_MON_RESET=1
        ct2.ALIGN_MON_RESET=0
        print(f'{b} {lane}: {ct2.ALIGN_MON_WORD:16b}')


In [ ]:
# ca.sync_async()
for b in ca.ib:
    ct2 = b.CT.CT2
    for lane in range(4):
        ct2.ALIGN_MON_SOURCE=7
        ct2.ALIGN_MON_LANE=lane
        ct2.ALIGN_MON_RESET=1
        ct2.ALIGN_MON_RESET=0
        time.sleep(.1)
        print(f'{b} {lane}: {ct2.ALIGN_MON_WORD}')


In [ ]:
for b in ca.ib:
    ct2 = b.CT.CT2
    ct2.ALIGN_MON_RESET=1
    ct2.ALIGN_MON_RESET=0
ct2.ALIGN_IGNORE_LANE


In [ ]:
for b in ca.ib:
    ct2 = b.CT.CT2
    ct2.ALIGN_SOF_WINDOW=70


In [ ]:
await ca.sync_async()
ca.ib.REFCLK.SYNC_CTR

In [ ]:
for b in ca.ib:
    ct1 = b.CT.CT1
    print(f'{ct1} {ct1.CT1_RST_STATUS=}, {ct1.ARST_STATUS=}, {ct1.RST_STATUS=}')

In [ ]:
source = 1
for b in ca.ib:
    ct2 = b.CT.CT2
    ct3 = b.CT.CT3
    for lane in range(4):
        ct2.ALIGN_MON_SOURCE=source
        ct2.ALIGN_MON_LANE=lane
        ct3.ALIGN_MON_SOURCE=source
        ct3.ALIGN_MON_LANE=lane
        ct2.ALIGN_MON_RESET=1
        ct3.ALIGN_MON_RESET=1
        ct2.ALIGN_MON_RESET=0
        ct3.ALIGN_MON_RESET=0
        print(f'{b} {lane}: CT2: {ct2.ALIGN_MON_WORD:5d}  CT3: {ct3.ALIGN_MON_WORD:5d}')

In [ ]:
ct2=ca.ib[0].CT.CT2

In [ ]:
ct2.LANE_MAP_BYTE0

In [ ]:
ct2.LA

In [ ]:
ct2.ALIGN_SOF_WINDOW

In [ ]:
await ca.sync_async()

In [ ]:
ct2.ALIGN_MON_LANE=3
set(ct2.ALIGN_MON_WORD for _ in range(10000))

In [ ]:
[b.pll.get_active_clkin() for b in ca.ib]
    

In [ ]:
ca.get_sync_method()

In [ ]:
# Sinewaves

In [ ]:
bins = np.zeros(8192, dtype=np.int32)
# bins[2000:2100:5] = 1 # shows sidebands
bins[[1999,2015, 2023]] = 1 # shows sidebands
nonzero_bins, = np.where(bins)
t=np.arange(16384)
buffer=2000*np.sin(2*np.pi*nonzero_bins[:,None]*t[None,:]/len(t)).sum(axis=0)
if buffer.max() > (2**13)-1 or buffer.min() < -2**13:
    raise RuntimeError('Waveform exceeds 14-bit range')
in_map = np.zeros(8192, bool)
in_map[bm.flatten()]=True
not_in_map = ~in_map
ignored_bins, = np.where((bins!=0) & not_in_map)

if ignored_bins.any():
    print(f"{len(ignored_bins)}/{len(nonzero_bins)} bins ({ignored_bins}) are not in the bin map and won't show up in the graph")
for b in ca.ib:
    b.set_channelizer(data_source='buffer', buffer=buffer, fft_bypass=False, scaler_bypass=False, offset_binary_encoding=False, gains=1<<29) 
await ca.sync_async()

In [ ]:
in_map

In [ ]:
16664/8

In [ ]:
.36/2/.027

In [ ]:
fig, ax = plt.subplots()
ax.set_title('FFT Input')
ax.plot(buffer)

In [ ]:
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush=True, soft_integ_period=1, number_of_results=3, verbose=1)
dd0_clean = dd0
dd0_clean[:,bad_bins,:] = 0
mags = abs(dd0_clean[0,:,0])

In [ ]:
fig, ax = plt.subplots()
ax.set_title('Autocorr CH0, sum of sinewaves through FFT')
ax.plot(mags)
ax.plot(bins*integ_period, '-g')
ax.plot(bins*integ_period*in_map, '.b')
# ax.plot(np.where(in_map)[0], mags[in_map], 'b.')
ax.plot(bad_bins, mags[bad_bins], 'r.')

In [ ]:
# Check if all bins that are set to 1+0j yield the expected integrated value
expected_mags = np.zeros(8192)
expected_mags = np.sign(bins * in_map)**2 * integ_period # oly bins in the bin map shall have the integration value
expected_mags[bad_bins] = 0 # ... except the bad bins that are coopted by the correlator for its read cycles
print(f'Got the expected spectrum: {all(np.sign(mags)*integ_period == expected_mags)}')
np.where(np.sign(mags)*integ_period != expected_mags)

In [ ]:
bins[2052]

In [ ]:
in_map[2052]

In [ ]:
mags[2052]

In [ ]:
((bins*in_map)**2*integ_period)[2052]

In [ ]:
bad_bins

In [ ]:
np.sign(-3)

In [ ]:
# Sinewave ramp

In [ ]:
# Generate dict of value: (i,j), where value are unique non-zero abs(a+jb)**2) and a & b are 4-bit signed values 
vv=list({int(abs(a+1j*b)**2): (a,b) for a in range(8) for b in range(8)}.items())
v2=[vv[ii][1] for ii in sorted(range(len(vv)), key=lambda k:vv[k][0]) if vv[ii][0]]
in_map = np.zeros(8192, bool)
in_map[bm.flatten()]=True
in_map_bins, = np.where(in_map)
bins = np.zeros(8192, dtype=np.complex64)
# bins[2000:2100:5] = 1 # shows sidebands
# bins[1:8192] = np.linspace(8000, 2000, 8191) # shows sidebands
bins[in_map_bins] = np.array([v2[ix][0]+1j*v2[ix][1] for ix in np.arange(len(in_map_bins)) % len(v2)])

# bins = np.array([v2[ix][0]+1j*v2[ix][1] for ix in np.arange(8192) % len(v2)])
nonzero_bins, = np.where(bins)
t=np.arange(16384)/16384
w=2*np.pi*np.arange(8192)
buffer=(bins[:,None].real * np.cos(w[:,None]*t[None,:]) + bins[:,None].imag * np.sin(w[:,None]*t[None,:])).sum(axis=0)
buffer = buffer/buffer.max() * 8191.0
# if buffer.max() > (2**13)-1 or buffer.min() < -2**13:
#     raise RuntimeError('Waveform exceeds 14-bit range')
not_in_map = ~in_map
ignored_bins, = np.where((bins!=0) & not_in_map)

if ignored_bins.any():
    print(f"{len(ignored_bins)}/{len(nonzero_bins)} bins ({ignored_bins}) are not in the bin map and won't show up in the graph")
for b in ca.ib:
    b.set_channelizer(data_source='buffer', buffer=buffer, fft_bypass=False, scaler_bypass=False, offset_binary_encoding=False, gains=1.15 * 2**28) 
await ca.sync_async()

In [ ]:
fig, ax = plt.subplots()
ax.set_title('FFT Input')
ax.plot(abs(bins))
fig, ax = plt.subplots()
ax.set_title('FFT Input')
ax.plot(buffer)

In [ ]:
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush=True, soft_integ_period=1, number_of_results=3, verbose=1)
dd0_clean = dd0
dd0_clean[:,bad_bins,:] = 0
mags = abs(dd0_clean[0,:,0])

In [ ]:
# Check if all bins that are set to 1+0j yield the expected integrated value
expected_mags = np.zeros(8192)
expected_mags = (abs(bins)**2 * in_map).round() # oly bins in the bin map shall have the integration value
expected_mags[bad_bins] = 0 # ... except the bad bins that are coopted by the correlator for its read cycles
match = mags == expected_mags
print(f'Got the expected spectrum: {all(match)}')
np.where(mags != expected_mags)

In [ ]:
b=np.arange(8192)
fig, ax = plt.subplots()
ax.set_title('Autocorr CH0, sum of sinewaves through FFT')
ax.plot(b[in_map], expected_mags[in_map], 'oy')
ax.plot(b[in_map], mags[in_map], '.-b')
# ax.plot(bins**2, '-g')
# ax.plot(bins*integ_period*in_map, '.b')
# ax.plot(np.where(in_map)[0], mags[in_map], 'b.')
ax.plot(bad_bins, mags[bad_bins], 'r.')


In [ ]:
2057-1024

In [ ]:
i=8187
print(f'{bins[i]}, {expected_mags[i]}, {mags[i]}')

In [ ]:
expected_mags[263]

In [ ]:
mags[263]

# Tests with FFT_BYPASS=TRUE

In [ ]:
integ_period=65536 # 16384 is 500 Mbps total from all 4 boards, seems to have packet loss.
for b in ca.ib:
    b.set_channelizer(fft_bypass=True, scaler_bypass=False, offset_binary_encoding=False) 
await ca.start_correlators_async(integ_period)
cr = await ca.get_corr_receiver_async()

In [ ]:
bm=cr.bin_map
bad_bins = bm[:,0,0]
gain = 1<<((48-4)-(32-14)) # gain to move the lsb of 14-bit value stored at the tp of a 32-bit pre-gain number into the LSB of a 4 bit value stored at the top of a 48-bit post-gain number
# for b in ca.ib:
#     b.set_channelizer(data_source='zero', fft_bypass=True, scaler_bypass=False, offset_binary_encoding=False, gains=0) 
# ca.sync_async()    

## All Ones

In [ ]:
bins = np.ones(8192, dtype=np.complex64)

nonzero_bins, = np.where(bins)
not_in_map = np.isin(nonzero_bins, bm.flatten(), invert=True)
in_map = ~not_in_map
if any(not_in_map):
    print(f"The bins {nonzero_bins[not_in_map]} are not in the bin map and won't show up in the graph")
for b in ca.ib:
    b.set_channelizer(data_source='buffer', buffer=bins.view(np.float32), fft_bypass=True, scaler_bypass=False, offset_binary_encoding=False, gains=gain) 
await ca.sync_async()


In [ ]:
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush=True, soft_integ_period=1, number_of_results=3, verbose=1)
dd0_clean = dd0
dd0_clean[:,bad_bins,:] = 0
mags = abs(dd0_clean[0,:,0])

In [ ]:
in_map

In [ ]:
mag[in_map]

mag.shape

In [ ]:

fig, ax = plt.subplots()
ax.set_title('Autocorr CH0')
ax.plot(mags)
ax.plot(np.where(in_map)[0], mags[in_map], 'b.')
ax.plot(bad_bins, mags[bad_bins], 'r.')


In [ ]:
# Check if all bins that are set to 1+0j yield the expected integrated value
expected_mags = np.zeros(8192)
expected_mags[in_map] = integ_period # oly bins in the bin map shall have the integration value
expected_mags[bad_bins] = 0 # ... except the bad bins that are coopted by the correlator for its read cycles
all(mags == expected_mags)

## 1/0 Bin patterns

In [ ]:
np.where(bm==270)

In [ ]:
for b in ca.ib:
    b.CORR.SOFT_RESET=0

In [ ]:
bm,pm=ca.ib[0].CORR.get_bin_map()
bad_bins=bm[:,:,0,0]
bins = np.zeros(8192, dtype=np.complex64)
bins[bm[2]] = 1
nonzero_bins, = np.where(bins)

in_map = np.zeros(8192, bool)
in_map[bm.flatten()]=True
not_in_map = ~in_map
ignored_bins, = np.where((bins!=0) & not_in_map)
if ignored_bins.any():
    print(f"Ignoring {len(ignored_bins)}/{len(nonzero_bins)} bins that are not in the bin map and won't show up in the graph. \n Ignored bins are ({ignored_bins})")
for b in ca.ib:
    b.set_channelizer(data_source='buffer', buffer=bins.view(np.float32), fft_bypass=True, scaler_bypass=False, offset_binary_encoding=False, gains=gain) 
await ca.sync_async()

In [ ]:
# ca.sync_async()
tt0,dd0,cc0,ss0 = cr.read_corr_frames(flush=True, soft_integ_period=1, number_of_results=8, verbose=1)
dd0_clean = dd0
dd0_clean[:,bad_bins,:] = 0
mags = abs(dd0_clean[0,:,0])

In [ ]:
abs(dd0[:,:,0]).sum(axis=1)

In [ ]:
fig, ax = plt.subplots()
ax.set_title('Autocorr CH0, some bins set')
ax.plot(mags)
ax.plot(np.where(in_map)[0], mags[in_map], 'b.')
ax.plot(bad_bins, mags[bad_bins], 'r.')

In [ ]:
# Check if all bins that are set to 1+0j yield the expected integrated value
expected_mags = np.zeros(8192)
expected_mags[in_map] = abs(bins[in_map])**2 * integ_period # oly bins in the bin map shall have the integration value
expected_mags[bad_bins] = 0 # ... except the bad bins that are coopted by the correlator for its read cycles
all(mags == expected_mags)

In [ ]:
not_in_map

# Mapping exploration snippets

In [ ]:
N=4
z = np.empty((N,N), dtype=np.intp)
i,j=np.triu_indices(N)
z[i,j]=np.arange(len(i))
z[j,i]=np.arange(len(i))
print(z)
print(z.nbytes)

In [ ]:
NCLK=1
ij_prod_map = np.array([[(i, (j+i) % N) if i < N-j else ((j+i) % N, i-1)
                       for i in range(N+1) for j in range(N//2//NCLK*clk, N//2//NCLK*(clk+1)) ]
                       for clk in range(NCLK) ])


In [ ]:
z[ij_prod_map[...,0],ij_prod_map[...,1]]

In [ ]:
k=np.array([(0,0),(1,1),(2,2),(3,3)])
z[tuple(k.T)]

In [ ]:
tuple(k.T)

In [ ]:
dd0.shape

In [ ]:
cr.NPROD_PER_CMAC*cr.NCORRS

In [ ]:
fft_bin_map = np.arange(8192).reshape((4, -1), order='F')
fft_bin_map

In [ ]:
# Scrapbook

In [ ]:
a=np.arange(10, dtype=np.complex128)
a.view(np.float64)

In [ ]:
bm.shape

In [ ]:
[abs(x)**2*32768 for x in (1, 1+1j, 1-1j, 2, 2+1j, 2-1j, 1+2j, 1-1j, 2+2j)]

In [ ]:
V=list(vv.items())
ix=

In [ ]:
vv=list({int(abs(a+1j*b)**2): (a,b) for a in range(-8,1) for b in range(-8,1)}.items())
v2={vv[ii][0]:vv[ii][1] for ii in sorted(range(len(vv)), key=lambda k:vv[k][0]) if vv[ii][0]}

In [ ]:
fig, ax = plt.subplots()
ax.plot(v2.keys(), '.-')

In [ ]:
vv

In [ ]:
(3+0j)**(1/6)

In [ ]:
a=np.full(10,0xff, np.uint8)

In [ ]:
a[1] &= ~(1<<3)

In [ ]:
a

In [ ]:
bin(a[1])

In [ ]:
ca.ib.get_temperatures()

In [ ]:
bm = ca.ib[0].chan[0].FFT.get_bin_map()

In [ ]:
bm[:,:16]

In [ ]:
bm2 = np.array([[b for clk,b in enumerate(lane_bins) if not (clk < 256 and clk & 0b11 == lane)] for lane, lane_bins in enumerate(bm)])


In [ ]:
bm2[:,:16]

In [ ]:
bm3 = np.full((8, 1984//2), -1, dtype=np.int32)
for lane, bins in enumerate(bm2):
    for bix, b in enumerate(bins):
        bm3[lane + 4*(bix & 1), bix//2] = b
bm3[:,:16]

In [ ]:
for b in ca.ib:
    print(b.CT.CT3.SLOT_GROUP)

In [ ]:
bm4 = bm3[:,7::8]
bm4[:, :16]

In [ ]:
bm=cr.bin_map

In [ ]:
bm[:,:16,0]

In [ ]:
bm,pm=ca.ib[0].CORR.get_bin_map()

In [ ]:
bm[:,:16*16:16,0]

In [ ]:
bms=bm[:,::16,0] 
bms[:,:16]

In [ ]:
all((bms==bm4).flatten())

In [ ]:
bms==bm4

In [ ]:
bm2[:,:16]

In [ ]:
bm21=bm2.reshape(4,2,-1, order='F')
bm21[:,:16]

In [ ]:
bm22=bm21.transpose((1,0,2)).reshape(8,-1)
bm22[:,:16]

In [ ]:
bm22=bm21.reshape(8,-1)
bm22[:,:16]

In [ ]:
bm23=bm22[:,7::8]
bm23[:,:16]

In [ ]:
bm22[[0,4,1,5,2,6,3,7]]

In [ ]:
x=[1,2,1]
y=[4,5,4]
set(zip(x,y))

In [ ]:
a=np.array([1,2,3])

In [ ]:
b=a[2:3]

In [ ]:
b[0]=4
a

In [ ]:
b.shape

In [ ]:
a=np.array(2)

In [ ]:
a.shape

In [ ]:
a[()]=3

In [ ]:
a

In [ ]:
a=np.arange(10)+10
b=np.arange(20)+20
x = a[:10]
c=x[3:5]
x=b[:10]

In [ ]:
x

In [ ]:
c

In [ ]:
def x(s):
    print(s)

In [ ]:
x(slice(None))

In [ ]:
c[slice(None)]

In [ ]:
# Send the reboot command to TCPIPE
import socket
for mb in ca.ib:
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as sock:
        sock.sendto(b"REBOOT", (mb.hostname, 9875))